# Machine Learning Capstone Project - Regression Track

This notebook covers the Regression track of the Capstone project. It includes EDA, Preprocessing, and the implementation of multiple regression algorithms.

## Section A - Dataset & EDA

**Note:** We are using the California Housing dataset as a placeholder. Please replace it with the assigned dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

# Set seed for reproducibility
np.random.seed(42)

# Load placeholder dataset
# TODO: Replace with pd.read_csv('data/your_dataset.csv')
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Target'] = data.target

# Dataset Audit
print("Dataset Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Distribution:\n", df['Target'].describe())

### EDA Visualisations
Visualising the distribution of the target variable and features.

In [ ]:
# Target Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Target'], bins=50, kde=True)
plt.title('Distribution of Target Variable')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

# Scatter plots for selected features vs target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(ax=axes[0], data=df, x='MedInc', y='Target', alpha=0.5)
axes[0].set_title('Median Income vs Target')
sns.scatterplot(ax=axes[1], data=df, x='HouseAge', y='Target', alpha=0.5)
axes[1].set_title('House Age vs Target')
plt.show()

### Insight Commentary
- **Target Distribution**: The target variable appears somewhat normally distributed but is right-skewed and capped at 5.0.
- **Correlation**: `MedInc` (Median Income) shows a strong positive correlation with the target variable.
- **Relationships**: The scatter plots confirm that higher median income generally leads to higher house values.

## Section B - Preprocessing & Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Data Cleaning
# (No missing values in California Housing, but here is a standard template)
# df.fillna(df.median(), inplace=True)
# df.drop_duplicates(inplace=True)

# 2. Feature Engineering
# Example: Creating a new feature representing rooms per household
df['RoomsPerHousehold'] = df['AveRooms'] / (df['AveOccup'] + 1e-6)

# 3. Splitting
X = df.drop('Target', axis=1)
y = df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Scaling (fitted on train set only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Testing set shape:", X_test_scaled.shape)

## Section C - Regression Track

We will now implement the regression models.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV, cross_val_score

# Dictionary to store results
results = {}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name} -> R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")
    return y_pred

### 6. Decision Tree Regressor
Tuning `max_depth` and showing feature importance.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
# Tuning max_depth
param_grid_dt = {'max_depth': [3, 5, 7, 10, None]}
grid_dt = GridSearchCV(dt, param_grid_dt, cv=5, scoring='r2')
grid_dt.fit(X_train_scaled, y_train)

best_dt = grid_dt.best_estimator_
print(f"Best parameters for Decision Tree: {grid_dt.best_params_}")

_ = evaluate_model("Decision Tree", best_dt, X_train_scaled, X_test_scaled, y_train, y_test)

# Feature Importance Plot
importances = best_dt.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title('Feature Importances - Decision Tree')
plt.show()

### 7. Random Forest Regressor
Ensemble baseline; tuning `n_estimators`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
# Tuning n_estimators
param_grid_rf = {'n_estimators': [50, 100, 200]}
grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='r2')
grid_rf.fit(X_train_scaled, y_train)

best_rf = grid_rf.best_estimator_
print(f"Best parameters for Random Forest: {grid_rf.best_params_}")

_ = evaluate_model("Random Forest", best_rf, X_train_scaled, X_test_scaled, y_train, y_test)

*(End of Day 1 work. Algorithms 8, 9, 10 will be added on subsequent days.)*